In [ ]:
# ==== One-cell E2E text-classification workflow with caching, CV, calibrated probs, threshold search, and final ensemble ====
# Files: train.csv (imbalanced ~70/30), val.csv (balanced-ish), test.csv (balanced-ish)
# Outputs: ./artifacts/*.joblib (model weights), ./artifacts/metrics.json, ./artifacts/test_preds_*.csv



In [ ]:
# ---------------------- CONFIG ----------------------
RANDOM_STATE = 42
N_JOBS = -1  # set to 1 if RAM-limited
VERBOSE = 1
USE_SUBSET = True
SUBSET_ROWS = 60000  # stratified sample from the imbalanced train
ARTIFACT_DIR = "artifacts"

In [ ]:
# Retrain toggles (if False and an artifact exists, it will load from disk)
RETRAIN_SGD = False
RETRAIN_LINSVC = False
RETRAIN_NB = False
RETRAIN_LOGREG = False
RETRAIN_KNN = False
RETRAIN_ENSEMBLE = False  # just recompute blend and thresholds; no weights to load

In [ ]:
# ---------------------- IMPORTS ----------------------
import os, json, math, warnings, re, sys, gc, textwrap, random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from joblib import dump, load

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.decomposition import TruncatedSVD
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
    classification_report, accuracy_score
)
from sklearn.utils import check_random_state
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

os.makedirs(ARTIFACT_DIR, exist_ok=True)



In [ ]:
# ---------------------- HELPERS (unique names) ----------------------
def fresh_seed_everywhere(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)

def fresh_quick_clean(txt):
    # very light normalization; keep it cheap to avoid changing signal
    if not isinstance(txt, str):
        txt = "" if pd.isna(txt) else str(txt)
    txt = re.sub(r"\s+", " ", txt.strip())
    return txt

def fresh_read_and_prepare(csv_path, text_col="text", label_col="label"):
    df = pd.read_csv(csv_path)
    if text_col not in df.columns:
        # try to guess text column if not provided
        candidates = [c for c in df.columns if df[c].dtype == object]
        text_col = candidates[0]
    if label_col not in df.columns and "label" in df.columns:
        label_col = "label"
    if label_col in df.columns:
        df[label_col] = df[label_col].astype(int)
    df[text_col] = df[text_col].astype(str).map(fresh_quick_clean)
    return df, text_col, label_col

def fresh_stratified_subset(X, y, max_rows=SUBSET_ROWS, seed=RANDOM_STATE):
    if len(y) <= max_rows:
        return X, y
    rs = check_random_state(seed)
    idx0 = np.where(y.values == 0)[0]
    idx1 = np.where(y.values == 1)[0]
    # keep class ratio; sample proportionally
    frac = max_rows / float(len(y))
    n0 = max(1, int(round(len(idx0) * frac)))
    n1 = max(1, int(round(len(idx1) * frac)))
    sel0 = rs.choice(idx0, size=n0, replace=False)
    sel1 = rs.choice(idx1, size=n1, replace=False)
    sel = np.concatenate([sel0, sel1])
    rs.shuffle(sel)
    return X.iloc[sel], y.iloc[sel]

def fresh_eval_binary(y_true, y_proba, thresh=0.5, prefix=""):
    y_pred = (y_proba >= thresh).astype(int)
    metrics = {
        f"{prefix}f1": f1_score(y_true, y_pred),
        f"{prefix}precision": precision_score(y_true, y_pred),
        f"{prefix}recall": recall_score(y_true, y_pred),
        f"{prefix}accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}auc": roc_auc_score(y_true, y_proba),
        f"{prefix}threshold": float(thresh)
    }
    return metrics

def fresh_sweep_threshold(y_true, y_proba, metric="f1", steps=401):
    # choose threshold maximizing requested metric; fall back to F1 if unknown
    thr_grid = np.linspace(0.0, 1.0, steps)
    best_t, best_m, best_pack = 0.5, -1.0, None
    for t in thr_grid:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred)
        if metric == "precision":
            m = prec
        elif metric == "recall":
            m = rec
        elif metric == "auc":
            m = roc_auc_score(y_true, y_proba)  # constant across t; included for completeness
        else:
            m = f1
        if m > best_m or (math.isclose(m, best_m) and abs(t-0.5) < abs(best_t-0.5)):
            best_m, best_t = m, t
            best_pack = dict(f1=f1, precision=prec, recall=rec)
    return float(best_t), best_m, best_pack

def fresh_print_eda(df, text_col, label_col=None, name="DF"):
    print(f"\n== EDA: {name} ==")
    print(df.head(3))
    print(f"Rows: {len(df):,}")
    lens = df[text_col].map(lambda s: len(s.split()))
    print(f"Token length -> mean {lens.mean():.1f}, median {lens.median():.1f}, 95p {np.percentile(lens,95):.1f}")
    if label_col and label_col in df.columns:
        counts = df[label_col].value_counts().sort_index()
        print("Label distribution:", counts.to_dict(), "| ratio 1's:", counts.get(1,0)/counts.sum() if counts.sum() else 0)

def fresh_save(obj, path):
    dump(obj, path)
    print(f"[saved] {path}")

def fresh_load(path):
    print(f"[loaded] {path}")
    return load(path)

def fresh_cv_search(name, pipe, grid, X, y, scoring="f1", cv=3, n_jobs=N_JOBS, verbose=VERBOSE):
    print(f"\n>>> GridSearchCV for {name}")
    gs = GridSearchCV(pipe, param_grid=grid, scoring=scoring, cv=cv, n_jobs=n_jobs, verbose=verbose)
    gs.fit(X, y)
    print("Best params:", gs.best_params_)
    print("Best CV score:", gs.best_score_)
    return gs.best_estimator_, gs.best_score_, gs.best_params_

In [ ]:
# ---------------------- READ DATA ----------------------
fresh_seed_everywhere(RANDOM_STATE)
train_df, TEXT_COL, LABEL_COL = fresh_read_and_prepare("train.csv", text_col="text", label_col="label")
val_df, _, _ = fresh_read_and_prepare("val.csv", text_col=TEXT_COL, label_col=LABEL_COL)
test_df, _, _ = fresh_read_and_prepare("test.csv", text_col=TEXT_COL, label_col=None)

fresh_print_eda(train_df, TEXT_COL, LABEL_COL, name="train.csv (imbalanced)")
fresh_print_eda(val_df, TEXT_COL, LABEL_COL, name="val.csv (balanced-ish)")
fresh_print_eda(test_df, TEXT_COL, None, name="test.csv")

X_train_full = train_df[TEXT_COL].astype(str)
y_train_full = train_df[LABEL_COL].astype(int)
X_val = val_df[TEXT_COL].astype(str)
y_val = val_df[LABEL_COL].astype(int)
X_test = test_df[TEXT_COL].astype(str)

# Optional stratified sub-sample for compute efficiency on CV
if USE_SUBSET:
    X_tr, y_tr = fresh_stratified_subset(X_train_full, y_train_full, max_rows=SUBSET_ROWS, seed=RANDOM_STATE)
else:
    X_tr, y_tr = X_train_full, y_train_full

print(f"\nTraining set used for model selection: {len(y_tr):,} rows | class prior ~ {y_tr.mean():.3f}")

In [ ]:
# ---------------------- MODEL DEFINITIONS ----------------------
art_sgd = f"{ARTIFACT_DIR}/sgd_hash_tfidf.joblib"
art_svc = f"{ARTIFACT_DIR}/linsvc_char_calibrated.joblib"
art_nb  = f"{ARTIFACT_DIR}/cnb_word_tfidf.joblib"
art_lr  = f"{ARTIFACT_DIR}/logreg_word_tfidf.joblib"
art_knn = f"{ARTIFACT_DIR}/knn_svd_char.joblib"

best_models = {}
cv_scores = {}

# 1) SGDClassifier (logistic) on HashingVectorizer + TfidfTransformer
sgd_pipe = Pipeline([
    ("hash", HashingVectorizer(analyzer="word", ngram_range=(1,2),
                               n_features=2**21, alternate_sign=False,
                               norm=None, dtype=np.float32)),
    ("tfidf", TfidfTransformer()),
    ("clf", SGDClassifier(loss="log_loss", alpha=1e-4, early_stopping=True, n_jobs=1, random_state=RANDOM_STATE, verbose=1))
], verbose=True)
sgd_grid = {
    "hash__ngram_range": [(1,1), (1,2), (1,3)],
    "clf__alpha": [1e-5, 1e-4, 1e-3],
}
if RETRAIN_SGD or not os.path.exists(art_sgd):
    sgd_best, sgd_cv, sgd_params = fresh_cv_search("SGD(log_loss)+Hashing", sgd_pipe, sgd_grid, X_tr, y_tr, scoring="f1", cv=3)
    fresh_save(sgd_best, art_sgd)
else:
    sgd_best = fresh_load(art_sgd); sgd_cv = np.nan; sgd_params = {}
best_models["sgd"] = sgd_best; cv_scores["sgd"] = float(sgd_cv) if not np.isnan(sgd_cv) else None

# 2) LinearSVC with character n-grams + calibrated probabilities
svc_base = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3,5), sublinear_tf=True, min_df=1)),
    ("clf", LinearSVC(C=1.0, random_state=RANDOM_STATE, verbose=1))
], verbose=True)

svc_grid = {
    "tfidf__ngram_range": [(3,5), (3,6)],
    "tfidf__min_df": [1, 2],
    "clf__C": [0.5, 1.0, 2.0]
}

if RETRAIN_LINSVC or not os.path.exists(art_svc):
    svc_best, svc_cv, svc_params = fresh_cv_search(
        "LinearSVC(char)", svc_base, svc_grid, X_tr, y_tr, scoring="f1", cv=3
    )
    # Calibrate probabilities on the WHOLE pipeline (no extra tfidf outside)
    svc_calib = CalibratedClassifierCV(
        base_estimator=svc_best, method="sigmoid", cv=3, n_jobs=N_JOBS
    )
    svc_calib.fit(X_tr, y_tr)
    fresh_save(svc_calib, art_svc)
    svc_best = svc_calib
else:
    svc_best = fresh_load(art_svc); svc_cv = np.nan
best_models["svc"] = svc_best


# 3) ComplementNB with word TF-IDF
nb_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="word", sublinear_tf=True, min_df=1, max_df=0.99, ngram_range=(1,2))),
    ("clf", ComplementNB(alpha=1.0))
], verbose=True)
nb_grid = {
    "tfidf__ngram_range": [(1,1), (1,2)],
    "tfidf__min_df": [1, 2],
    "tfidf__max_df": [0.95, 0.99],
    "clf__alpha": [0.5, 1.0, 2.0]
}
if RETRAIN_NB or not os.path.exists(art_nb):
    nb_best, nb_cv, nb_params = fresh_cv_search("ComplementNB(word)", nb_pipe, nb_grid, X_tr, y_tr, scoring="f1", cv=3)
    fresh_save(nb_best, art_nb)
else:
    nb_best = fresh_load(art_nb); nb_cv = np.nan
best_models["cnb"] = nb_best; cv_scores["cnb"] = float(nb_cv) if not np.isnan(nb_cv) else None

# 4) Logistic Regression tuned for sparse imbalance (with stronger max_iter as requested)
lr_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1,2), sublinear_tf=True, min_df=2, max_df=0.98)),
    ("clf", LogisticRegression(
        solver="saga", penalty="l2", C=2.0,
        class_weight="balanced", max_iter=1000, n_jobs=N_JOBS,
        random_state=RANDOM_STATE, verbose=1
    ))
], verbose=True)
lr_grid = {
    "tfidf__min_df": [1, 2],
    "tfidf__max_df": [0.95, 0.98],
    "tfidf__ngram_range": [(1,1), (1,2)],
    "clf__C": [1.0, 2.0, 3.0]
}
if RETRAIN_LOGREG or not os.path.exists(art_lr):
    lr_best, lr_cv, lr_params = fresh_cv_search("LogReg(word)", lr_pipe, lr_grid, X_tr, y_tr, scoring="f1", cv=3)
    fresh_save(lr_best, art_lr)
else:
    lr_best = fresh_load(art_lr); lr_cv = np.nan
best_models["logreg"] = lr_best; cv_scores["logreg"] = float(lr_cv) if not np.isnan(lr_cv) else None

# 5) KNN on reduced char TF-IDF (SVD) to keep it tractable
knn_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(3,5), sublinear_tf=True, min_df=2, max_df=1.0)),
    ("svd", TruncatedSVD(n_components=250, random_state=RANDOM_STATE)),
    ("clf", KNeighborsClassifier(n_neighbors=100, weights="distance", metric="euclidean", n_jobs=N_JOBS))
], verbose=True)
knn_grid = {
    "tfidf__ngram_range": [(3,5), (3,6)],
    "svd__n_components": [200, 300],
    "clf__n_neighbors": [50, 100],
}
if RETRAIN_KNN or not os.path.exists(art_knn):
    knn_best, knn_cv, knn_params = fresh_cv_search("KNN(SVD+char)", knn_pipe, knn_grid, X_tr, y_tr, scoring="f1", cv=3)
    fresh_save(knn_best, art_knn)
else:
    knn_best = fresh_load(art_knn); knn_cv = np.nan
best_models["knn"] = knn_best; cv_scores["knn"] = float(knn_cv) if not np.isnan(knn_cv) else None



In [ ]:
# ---------------------- VALIDATION EVALUATION ----------------------
def fresh_predict_proba(model, X):
    # Works for Pipeline or CalibratedClassifierCV pipelines
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:,1]
    # Some calibrated pipelines may have named step
    try:
        return model.named_steps["calib"].predict_proba(model.named_steps["tfidf"].transform(X))[:,1]
    except Exception:
        # fallback: decision_function -> sigmoid calibration (approx)
        if hasattr(model, "decision_function"):
            z = model.decision_function(X)
            # safe sigmoid
            return 1/(1+np.exp(-z))
        raise ValueError("Model lacks probability interface")



In [ ]:
val_results = {}
val_probas = {}
for key, mdl in best_models.items():
    p = fresh_predict_proba(mdl, X_val)
    t_opt, _, pack = fresh_sweep_threshold(y_val, p, metric="f1", steps=501)
    met = fresh_eval_binary(y_val, p, thresh=t_opt, prefix=f"{key}_")
    val_results[key] = {**met, "t_star": t_opt}
    val_probas[key] = p
    print(f"\n[{key}] AUC={met[f'{key}_auc']:.4f} | F1*={met[f'{key}_f1']:.4f} @ t={t_opt:.3f} | P={met[f'{key}_precision']:.4f}, R={met[f'{key}_recall']:.4f}")



In [ ]:
# ---------------------- ENSEMBLE (soft-vote with CV-weighting) ----------------------
# Use normalized CV scores as weights if available; otherwise equal weights
weights = {}
valid_keys = list(val_probas.keys())
cv_vals = np.array([cv_scores.get(k, np.nan) for k in valid_keys], dtype=float)
if np.all(np.isnan(cv_vals)):
    weights = {k: 1.0 for k in valid_keys}
else:
    # replace nans with mean of non-nans
    m = np.nanmean(cv_vals)
    cv_vals = np.where(np.isnan(cv_vals), m, cv_vals)
    # avoid negatives; shift to positive scale
    cv_vals = cv_vals - cv_vals.min() + 1e-6
    cv_vals = cv_vals / cv_vals.sum()
    weights = {k: cv_vals[i] for i,k in enumerate(valid_keys)}

print("\nEnsemble weights:", weights)

ens_proba_val = np.zeros_like(y_val, dtype=float)
for k, p in val_probas.items():
    ens_proba_val += weights[k] * p
# normalize weights sum to 1
ens_proba_val = ens_proba_val / sum(weights.values())
t_ens, _, _ = fresh_sweep_threshold(y_val, ens_proba_val, metric="f1", steps=801)
ens_metrics = fresh_eval_binary(y_val, ens_proba_val, thresh=t_ens, prefix="ens_")
val_results["ensemble"] = {**ens_metrics, "t_star": t_ens}
print(f"\n[Ensemble] AUC={ens_metrics['ens_auc']:.4f} | F1*={ens_metrics['ens_f1']:.4f} @ t={t_ens:.3f} | P={ens_metrics['ens_precision']:.4f}, R={ens_metrics['ens_recall']:.4f}")



In [ ]:
# ---------------------- FINAL REFIT ON TRAIN ∪ VAL ----------------------
# Two-stage: fit each model on concatenated data, then produce test predictions using the ensemble with validation-chosen weights and threshold.
X_train_all = pd.concat([X_train_full, X_val], axis=0).reset_index(drop=True)
y_train_all = pd.concat([y_train_full, y_val], axis=0).reset_index(drop=True)

def fresh_refit_or_load(base_path, builder_fn, retrain_flag=True):
    if retrain_flag or not os.path.exists(base_path):
        mdl = builder_fn()
        mdl.fit(X_train_all, y_train_all)
        fresh_save(mdl, base_path)
    else:
        mdl = fresh_load(base_path)
    return mdl

# Builders mirror the best parameters already found
def build_sgd_final():
    # reuse tuned structure; if artifact exists from CV step, simply load → but we refit on train_all
    if isinstance(best_models["sgd"], Pipeline):
        tuned = best_models["sgd"]
        # rebuild same structure to guarantee clean refit
        return Pipeline([
            ("hash", tuned.named_steps["hash"]),
            ("tfidf", tuned.named_steps["tfidf"]),
            ("clf", tuned.named_steps["clf"])
        ])
    return sgd_pipe

def build_svc_final():
    # use calibrated pipeline found above; rebuild to ensure fit on train_all
    tuned = best_models["svc"]
    if "calib" in getattr(tuned, "named_steps", {}):
        tfidf = tuned.named_steps["tfidf"]
        base_clf = tuned.named_steps["calib"].base_estimator
        calib = CalibratedClassifierCV(base_clf, method="sigmoid", cv=3, n_jobs=N_JOBS)
        return Pipeline([("tfidf", tfidf), ("calib", calib)])
    return svc_base

def build_nb_final():
    tuned = best_models["cnb"]
    return Pipeline([
        ("tfidf", tuned.named_steps["tfidf"]),
        ("clf", tuned.named_steps["clf"])
    ])

def build_lr_final():
    tuned = best_models["logreg"]
    return Pipeline([
        ("tfidf", tuned.named_steps["tfidf"]),
        ("clf", tuned.named_steps["clf"].set_params(max_iter=1000))
    ])

def build_knn_final():
    tuned = best_models["knn"]
    return Pipeline([
        ("tfidf", tuned.named_steps["tfidf"]),
        ("svd", tuned.named_steps["svd"]),
        ("clf", tuned.named_steps["clf"])
    ])

In [ ]:
sgd_final = fresh_refit_or_load(f"{ARTIFACT_DIR}/FINAL_sgd.joblib", build_sgd_final, retrain_flag=RETRAIN_SGD)
svc_final = fresh_refit_or_load(f"{ARTIFACT_DIR}/FINAL_svc.joblib", build_svc_final, retrain_flag=RETRAIN_LINSVC)
nb_final  = fresh_refit_or_load(f"{ARTIFACT_DIR}/FINAL_cnb.joblib", build_nb_final, retrain_flag=RETRAIN_NB)
lr_final  = fresh_refit_or_load(f"{ARTIFACT_DIR}/FINAL_logreg.joblib", build_lr_final, retrain_flag=RETRAIN_LOGREG)
knn_final = fresh_refit_or_load(f"{ARTIFACT_DIR}/FINAL_knn.joblib", build_knn_final, retrain_flag=RETRAIN_KNN)

# Validation check (post-refit) just to log consistency
post_val = {}
post_val["sgd"] = fresh_predict_proba(sgd_final, X_val)
post_val["svc"] = fresh_predict_proba(svc_final, X_val)
post_val["cnb"] = fresh_predict_proba(nb_final, X_val)
post_val["logreg"] = fresh_predict_proba(lr_final, X_val)
post_val["knn"] = fresh_predict_proba(knn_final, X_val)

ens_val_final = np.zeros_like(y_val, dtype=float)
for k, p in post_val.items():
    w = weights.get(k, 1.0/len(post_val))
    ens_val_final += w * p
ens_val_final /= sum(weights.values())
# Keep the previously selected ensemble threshold t_ens (fitted on earlier val pass)
val_final_metrics = fresh_eval_binary(y_val, ens_val_final, thresh=val_results["ensemble"]["t_star"], prefix="ens_final_")
print(f"\n[Sanity after refit] Ensemble AUC={val_final_metrics['ens_final_auc']:.4f} | F1@t*={val_final_metrics['ens_final_f1']:.4f}")



In [ ]:
# ---------------------- TEST INFERENCE ----------------------
proba_test = {}
proba_test["sgd"] = fresh_predict_proba(sgd_final, X_test)
proba_test["svc"] = fresh_predict_proba(svc_final, X_test)
proba_test["cnb"] = fresh_predict_proba(nb_final, X_test)
proba_test["logreg"] = fresh_predict_proba(lr_final, X_test)
proba_test["knn"] = fresh_predict_proba(knn_final, X_test)

ens_test = np.zeros(len(X_test), dtype=float)
for k, p in proba_test.items():
    w = weights.get(k, 1.0/len(proba_test))
    ens_test += w * p
ens_test /= sum(weights.values())

# Use validation-optimized thresholds for each model and ensemble
def fresh_write_preds(name, proba, thresh):
    df_out = pd.DataFrame({"id": np.arange(len(proba)), "prob": proba, "pred": (proba >= thresh).astype(int)})
    out_path = f"{ARTIFACT_DIR}/test_preds_{name}.csv"
    df_out.to_csv(out_path, index=False)
    print(f"[saved] {out_path}")

for k in ["sgd", "svc", "cnb", "logreg", "knn"]:
    fresh_write_preds(k, proba_test[k], val_results[k]["t_star"])
fresh_write_preds("ensemble", ens_test, val_results["ensemble"]["t_star"])



In [ ]:
# ---------------------- METRICS SUMMARY & SAVING ----------------------
summary = {
    "cv_scores": cv_scores,
    "val_results": val_results,
    "ensemble_weights": weights
}
with open(f"{ARTIFACT_DIR}/metrics.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\n== Summary ==")
print(json.dumps(summary, indent=2))

In [ ]:
# ---------------------- NOTES ----------------------
# 1) The logistic regression uses SAGA, L2, class_weight='balanced', and max_iter=1000 to ensure convergence on sparse TF-IDF.
# 2) LinearSVC is wrapped in CalibratedClassifierCV to yield probabilities for thresholding and ensembling.
# 3) KNN operates on an SVD-reduced char TF-IDF space to make distance computations tractable at scale.
# 4) Thresholds are selected on the validation set by maximizing F1; ROC-AUC is reported to check separability.
# 5) All model objects are persisted; toggles at the top control whether to retrain or load existing artifacts.
# 6) Final training is on train∪val; predictions for test with both per-model and ensemble outputs are saved.
